In [1]:
# !pip install mlxtend==0.17.2
!pip install --upgrade mlxtend
!pip show mlxtend

Name: mlxtend
Version: 0.23.4
Summary: Machine Learning Library Extensions
Home-page: https://github.com/rasbt/mlxtend
Author: 
Author-email: Sebastian Raschka <mail@sebastianraschka.com>
License: BSD 3-Clause
Location: /usr/local/lib/python3.11/dist-packages
Requires: joblib, matplotlib, numpy, pandas, scikit-learn, scipy
Required-by: 


In [2]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import itertools
import copy
import numpy as np
from scipy.stats import stats
import random
import os
import sys
from statistics import mean
from statistics import stdev
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import fpmax

In [ ]:
import pickle
import re
from mlxtend.preprocessing import TransactionEncoder

#save data to loc_file
def save_data(dataset, loc_file, pre_loc_file_url = '/content/drive/My Drive/4_Proposed_Approach/'):
  loc_file = pre_loc_file_url + loc_file
  with open(loc_file, 'wb') as filehandle:
      pickle.dump(dataset, filehandle)

#load data from loc_file
def load_data(loc_file, pre_loc_file_url = '/content/drive/My Drive/4_Proposed_Approach/'):
  loc_file = pre_loc_file_url + loc_file
  print(loc_file)
  with open(loc_file, 'rb') as filehandle:
      dataset = pickle.load(filehandle)
  return dataset

def remove_subset(list_of_list):
    sets={frozenset(e) for e in list_of_list}
    us=[]
    while sets:
        e=sets.pop()
        if any(e.issubset(s) for s in sets) or any(e.issubset(s) for s in us):
            continue
        else:
            us.append(sorted(list(e)))
    return us

def convert_transaction_to_df_one_hot_vector(_transaction_dataset, sparse_format = True):
  te = TransactionEncoder()
  if(sparse_format):
    te_ary = te.fit(_transaction_dataset).transform(_transaction_dataset, sparse=True)
    return pd.DataFrame.sparse.from_spmatrix(te_ary, columns=te.columns_)
  te_ary = te.fit(_transaction_dataset).transform(_transaction_dataset)
  return pd.DataFrame(te_ary, columns=te.columns_)

def print_list_with_minK(list_of_list, min_k_item):
  count = 0
  for item in list_of_list:
    if(len(item) >= min_k_item):
      count +=1
      print(item)
  print(f"Len -origin:{len(list_of_list)} ")
  print(f"Len -filter by min_k_item={min_k_item} :{count} ")

def is_binary_device_by_df(df):
    if (df['device_id'][0:1] == 'M'
        or df['device_id'][0:1] == 'D'
        or df['device_id'][0:2] == 'L0'):
        val = bool(True)
    else:
        val = bool(False)
    return val

def is_binary_device_by_id(device_id):
    if (device_id[0:1] == 'M'
        or device_id[0:1] == 'D'
        or device_id[0:2] == 'L0'):
        val = bool(True)
    else:
        val = bool(False)
    return val

def is_actuator_by_id(device_id):
  if re.search("[LD]\d+", device_id):
    return True
  return False

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
<>:66: DeprecationWarning: invalid escape sequence '\d'
<>:66: DeprecationWarning: invalid escape sequence '\d'
<ipython-input-4-c69e6cf027d6>:66: DeprecationWarning: invalid escape sequence '\d'
  if re.search("[LD]\d+", device_id):


In [ ]:
#read training data

# data = load_data('hh103_training.data')
# data = load_data('hh101_training.data')
data = load_data('hh102_training.data')

# take 1 year of data
start_date = data.iloc[0].datetime
end_date = start_date + datetime.timedelta(weeks=12)
data = data[data['datetime'].between(start_date, end_date)]

device_info = pd.DataFrame([], columns=['device_id', 'is_binary_device'])
device_info['device_id'] = data['device_id'].unique()
device_info['is_binary_device'] = device_info['device_id'].apply(lambda x: data[data['device_id'] == x]['is_binary_device'].iloc[0])

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


/content/drive/My Drive/4_Proposed_Approach/hh102_training.data


In [ ]:
devices = data['device_id'].unique()
list(filter(lambda x: is_actuator_by_id(x), devices))

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


['D002',
 'L003',
 'D005',
 'L002',
 'D006',
 'L005',
 'L004',
 'L001',
 'D001',
 'D003']

**Generate set of transaction which contains devices in same time windows**

In [ ]:
TIME_WINDOW_IN_MINUTES = 2
transaction_dataset = []

start_date = data.iloc[0].datetime

last_datetime =  data.iloc[-1].datetime

data_to_make_group_dev = data[(data['datetime'] < last_datetime)].copy(deep=True)
start = copy.deepcopy(start_date)
num_window = 0
while start<last_datetime:
    end = start + datetime.timedelta(minutes=(TIME_WINDOW_IN_MINUTES))
    data_extract = data_to_make_group_dev[(data_to_make_group_dev['datetime'] >= start) & (data_to_make_group_dev['datetime'] < end)]
    prev_items = list()
    if not data_extract.empty:
      current_items = []
      for k,v in data_extract.iterrows():
        if not v['device_id'] in current_items:
          current_items.append(v['device_id'])
      prev_items = current_items
      num_of_items = len(current_items)
      if(num_of_items>1):
        num_window +=1
        transaction_dataset.append(current_items)
      current_items = list()

    start = end

print(f'number of window: {num_window}')


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


number of window: 12861


**Learn frequent groups from the set of transactions by FPMAX algorithm**

In [ ]:
     from mlxtend.frequent_patterns import fpmax

# hh103
# actuators = ['D002','L005','L004','D001','L002','L003','D003']

# hh101
# actuators = []

# hh102
actuators = ['D002', 'L003', 'D005', 'L002', 'D006', 'L005', 'L004', 'L001', 'D001', 'D003']

df_transaction_dataset = convert_transaction_to_df_one_hot_vector(transaction_dataset)
rules = pd.DataFrame([], columns=['support', 'itemsets'])
for act_dev in actuators:
  data_extract = df_transaction_dataset[df_transaction_dataset[act_dev] == 1]
  # print(data_extract)
  if (len(data_extract) < 100):
    # print(act_dev)
    # print(len(data_extract))
    continue
  min_sup = 0.9
  while min_sup <= 1:
    frequent_itemset = fpmax(data_extract, min_support=min_sup, use_colnames=True, verbose=0)
    print("Frequent itemset: ", frequent_itemset)
    if ((frequent_itemset.itemsets.size < 20) or (min_sup == 1)):
      #rules.append(frequent_itemset)
      rules = pd.concat([rules, frequent_itemset], ignore_index=True)
      break
    min_sup = min_sup + 0.05
  #break
rules

Frequent itemset:      support             itemsets
0  0.924419  (M002, MA003, D002)
1  0.932171  (M001, MA003, D002)
Frequent itemset:      support                    itemsets
0  0.951389  (MA013, L003, LS013, M018)
Frequent itemset:     support            itemsets
0      1.0  (M008, M007, D005)
Frequent itemset:      support                           itemsets
0  0.934426  (MA003, LS009, MA009, L002, M002)


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/mlxtend/frequent_patterns/fpcommon.py:110: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/mlxtend/frequent_patterns/fpcommon.py:110: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/mlxtend/frequent_patterns/fpcommon.py

Frequent itemset:      support             itemsets
0  0.905983  (L004, M017, MA014)
1  0.905983  (L004, LS014, M017)
2  0.905983  (L004, LS014, M018)
3  0.923077   (L004, M017, M018)


,support,itemsets
0,0.924419,"(M002, MA003, D002)"
1,0.932171,"(M001, MA003, D002)"
2,0.951389,"(MA013, L003, LS013, M018)"
3,1.000000,"(M008, M007, D005)"
4,0.934426,"(MA003, LS009, MA009, L002, M002)"
5,0.905983,"(L004, M017, MA014)"
6,0.905983,"(L004, LS014, M017)"
7,0.905983,"(L004, LS014, M018)"
8,0.923077,"(L004, M017, M018)"


In [ ]:
data['device_id'].unique()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


array(['T101', 'MA009', 'MA007', 'MA008', 'LS003', 'LS004', 'LS007',
       'LS009', 'LS002', 'LS005', 'LS001', 'LS010', 'LS006', 'MA004',
       'L001', 'LS008', 'T103', 'M002', 'M001', 'D002', 'MA003', 'M005',
       'L005', 'MA006', 'T102', 'MA010', 'L004', 'D001', 'L002', 'L003',
       'D003'], dtype=object)

**Collect data points corresponding to resulting rules**

In [ ]:
# Gen transaction dataset
import math

def gen_datapoint(itemset):
  TIME_WINDOW_IN_MINUTES = 2
  start_date = data.iloc[0].datetime
  last_datetime =  start_date + datetime.timedelta(hours=24*90)#data.iloc[-1].datetime

  data_to_make_group_dev = data[(data['datetime'] < last_datetime)].copy(deep=True)
  start = copy.deepcopy(start_date)
  prev_datapoint = [0]*len(itemset)
  pointlist = []
  keys = list(device_info['device_id'])
  mapping = {key: -1 for key in keys}
  while start<last_datetime:
    #print(start)
    end = start + datetime.timedelta(minutes=(TIME_WINDOW_IN_MINUTES))
    data_extract = data_to_make_group_dev[(data_to_make_group_dev['datetime'] >= start) & (data_to_make_group_dev['datetime'] < end)]
    if not data_extract.empty:
      datapoint = []
      check = True
      for item in itemset:
        numRec = data_extract[data_extract['device_id'] == item].shape[0]
        value = 0
        if numRec == 0:
          value = mapping[item]
        else:
          value = math.ceil(data_extract[data_extract['device_id'] == item]['device_value'].mean())
          mapping[item] = data_extract[data_extract['device_id'] == item].iloc[-1].device_value
        #print(value)
        if (value == -1):
          check = False
          break
        if device_info[device_info['device_id'] == item]['is_binary_device'].iloc[0] == True:
          if value > 0:
            value = 50
          else:
            value = 0
        datapoint.append(value)

      if (check == True):
        pointlist.append(datapoint)
    start = end
  return pointlist

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
import collections

rules['datapoint'] = rules['itemsets'].apply(lambda x: collections.Counter(map(tuple,gen_datapoint(list(x)))))

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
# remove repeated data points
input = []
for index, row in rules.iterrows():
  inp = []
  for item in row['datapoint'].items():
    list_it = []
    for it in item[0]:
      list_it.append(it)
    inp.append(list_it)
  input.append(inp)
rules['input'] = input

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


**Train k-NearestNeighbors models corresponding to resulting rules**

In [ ]:
from sklearn.neighbors import NearestNeighbors

model_column = []
threshold_column = []
percentile_limit = 99.9

for index, row in rules.iterrows():
  X = row['input']
  try:
    nbrs = NearestNeighbors(n_neighbors=4, algorithm='ball_tree', metric='mahalanobis', metric_params={'V': np.cov(X,rowvar=False)}).fit(X)
  except:
    nbrs = NearestNeighbors(n_neighbors=4, algorithm='ball_tree').fit(X)
  distances, indices = nbrs.kneighbors(X)
  print("Distances", distances)
  dist_val = distances[:,1]
  print("Dist val", dist_val)
  dist_val = np.sort(dist_val)
  threshold = np.percentile(dist_val, percentile_limit)
  model_column.append(nbrs)
  threshold_column.append(threshold)

rules['model'] = model_column
rules['threshold'] = threshold_column

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Distances [[0.         1.87082869 1.87082869 1.87082869]
 [0.         1.87082869 1.87082869 1.87082869]
 [0.         1.87082869 1.87082869 1.87082869]
 [0.         1.87082869 1.87082869 1.87082869]
 [0.         1.87082869 1.87082869 1.87082869]
 [0.         1.87082869 1.87082869 1.87082869]
 [0.         1.87082869 1.87082869 1.87082869]
 [0.         1.87082869 1.87082869 1.87082869]]
Dist val [1.87082869 1.87082869 1.87082869 1.87082869 1.87082869 1.87082869
 1.87082869 1.87082869]
Distances [[0.         1.87082869 1.87082869 1.87082869]
 [0.         1.87082869 1.87082869 1.87082869]
 [0.         1.87082869 1.87082869 1.87082869]
 [0.         1.87082869 1.87082869 1.87082869]
 [0.         1.87082869 1.87082869 1.87082869]
 [0.         1.87082869 1.87082869 1.87082869]
 [0.         1.87082869 1.87082869 1.87082869]
 [0.         1.87082869 1.87082869 1.87082869]]
Dist val [1.87082869 1.87082869 1.87082869 1.87082869 1.87082869 1.87082869
 1.87082869 1.87082869]
Distances [[0.         0.0

In [ ]:
for i in range(len(rules['itemsets'])):
  rules['itemsets'][i] = list(rules['itemsets'][i])

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
<ipython-input-28-34fa993d2475>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rules['itemsets'][i] = list(rules['itemsets'][i])


In [ ]:
rules['itemsets'][0]

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


['LS001', 'D002', 'M002', 'LS002', 'LS007', 'MA007', 'M001']

In [ ]:
save_data(rules, 'hh102_greedy_v16.model')
# save_data(rules, 'hh102_v1.model')

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
